
# Step 0 — Precision Agriculture Patent Retrieval

## 1. Objective and scope

This notebook develops a reproducible patent retrieval
strategy for the Precision Agriculture sector, with
John Deere as the reference company.

The analysis focuses on crop production technologies,
including positioning, sensing, monitoring, guidance,
decision support, and precision application systems.

The search strategy is primarily informed by Tey
et al. (2024), *Evolution of precision agricultural
technologies: a patent network analysis*.

Avola et al. (2024) provides additional methodological
support for distinguishing precision-agriculture
innovations from generic agricultural machinery
improvements.

**Study period:** 2000–2022

**Unit of analysis:** DOCDB patent family

Patent families are treated as candidate sector-related
inventions. Their relevance will subsequently be
assessed through manual validation.

## Search strategy — three routes

- **R1:** specific CPC/IPC (including `A01B79/005` and `A01B69/008`), direct inclusion.
- **R2:** explicit Precision Agriculture terminology; 
- **R3:** **agricultural CPC/IPC AND enabling technology AND PA function**.

The enabling technologies and PA functions combine the previous R3 and R4 vocabularies; there are no technology–function pair restrictions and no additional agricultural-text requirement.

**Excluded classifications:** previous CPC/IPC filters apply only to R3. **Topical exclusions:** title-based terms for forestry, golf/turf, livestock and ornamental gardening; protected `A01B79/005` families are never removed by these topical exclusions.

All evidence is aggregated at DOCDB family level (2000–2022); candidate relevance still requires manual validation. References: Tey (2024), Avola (2024), Rejeb (2024).

In [1]:

from pathlib import Path
import re

import pandas as pd
import yaml

from sqlalchemy import and_, or_, func, false, case

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)

from IPython.display import display


In [31]:

# Locate the project from its root or notebooks folder
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "config" / "sector_PA_FINAL.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "sector_PA_FINAL.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}"
    )

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

# PATSTAT settings
ENV = config["patstat"]["environment"]
IPR_TYPE = config["patstat"]["ipr_type"]

# Temporal scope
YEAR_MIN = int(config["analysis"]["year_min"])
YEAR_MAX = int(config["analysis"]["year_max"])

FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

# Keywords
SPECIFIC_PHRASES = config["keywords"]["specific"]
AGRICULTURAL_TERMS = config["keywords"]["agricultural"]
ENABLING_TERMS = config["keywords"]["enabling_technologies"]
PA_FUNCTION_TERMS = config["keywords"]["pa_functions"]
EXCLUSION_TERMS = config["keywords"].get("exclusion_terms", [])

# Patent classifications
SPECIFIC_CPC = config["cpc"]["specific"]
AGRICULTURAL_CPC = config["cpc"]["agricultural_context"]
SPECIFIC_IPC = config["ipc"].get("specific", [])
AGRICULTURAL_IPC = config["ipc"]["agricultural_context"]

EXCLUDE_CPC = config["cpc"].get("exclude", [])
EXCLUDE_IPC = config["ipc"].get("exclude", [])

# Validation
SAMPLE_SIZE = int(config["validation"]["sample_size"])
RANDOM_SEED = int(config["validation"]["random_seed"])

if YEAR_MIN > YEAR_MAX:
    raise ValueError("Invalid year range.")

if FAMILY_TIME_COLUMN != "earliest_filing_year":
    raise ValueError("Expected earliest_filing_year.")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "step0"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:", CONFIG_PATH)
print("Environment:", ENV)
print("Period:", YEAR_MIN, "-", YEAR_MAX)
print("Output directory:", OUTPUT_DIR)


Configuration: /home/jovyan/team-38-PA/config/sector_PA_FINAL.yaml
Environment: PROD
Period: 2000 - 2022
Output directory: /home/jovyan/team-38-PA/notebooks/outputs/step0



## 4. Keyword and classification matching

Keyword expressions are converted into case-insensitive
regular expressions.

The asterisk (*) is interpreted as a word-ending wildcard,
while spaces, hyphens, and underscores are accepted as
alternative separators.

Keywords are searched in the available patent titles
and abstracts.

For computational efficiency, the terms belonging to
the same keyword group are combined into a single
regular expression.

CPC and IPC classifications are matched using
normalized code prefixes.


In [3]:

def term_to_regex(term):
    """Convert a YAML keyword into a word-bounded regex."""
    term = term.strip().lower()

    if not term:
        raise ValueError("Search terms cannot be empty.")

    parts = re.split(r"[\s\-_]+", term)

    patterns = [
        re.escape(part).replace(
            r"\*", r"[a-zA-Z0-9]*"
        )
        for part in parts
        if part
    ]

    return (
        r"(?:\b|_)"
        + r"[\s\-_/]+".join(patterns)
        + r"(?:\b|_)"
    )


def keyword_condition(patterns):
    """Match any keyword in the combined title and abstract."""
    if not patterns:
        return false()

    doc_text = func.lower(
        func.concat(
            func.coalesce(
                TLS202_APPLN_TITLE.appln_title, ""
            ),
            " ",
            func.coalesce(
                TLS203_APPLN_ABSTR.appln_abstract, ""
            ),
        )
    )

    combined_pattern = "|".join(
        f"(?:{pattern})"
        for pattern in patterns.values()
    )

    return func.regexp_contains(
        doc_text,
        combined_pattern
    )


def symbol_prefix_condition(column, prefixes):
    """Match CPC/IPC prefixes after removing spaces."""
    if not prefixes:
        return false()

    symbol = func.upper(
        func.replace(column, " ", "")
    )

    return or_(*[
        symbol.like(
            prefix.replace(" ", "").upper() + "%"
        )
        for prefix in prefixes
    ])


SPECIFIC_PATTERNS = {
    term: term_to_regex(term)
    for term in SPECIFIC_PHRASES
}

AGRICULTURAL_PATTERNS = {
    term: term_to_regex(term)
    for term in AGRICULTURAL_TERMS
}

ENABLING_PATTERNS = {
    term: term_to_regex(term) for term in ENABLING_TERMS
}

PA_FUNCTION_PATTERNS = {
    term: term_to_regex(term) for term in PA_FUNCTION_TERMS
}

EXCLUSION_PATTERNS = {
    term: term_to_regex(term) for term in EXCLUSION_TERMS
}

print("Specific phrases:", len(SPECIFIC_PATTERNS))
print("Enabling technologies:", len(ENABLING_PATTERNS))
print("PA functions:", len(PA_FUNCTION_PATTERNS))
print("Topical exclusions:", len(EXCLUSION_PATTERNS))

Specific phrases: 7
Enabling technologies: 51
PA functions: 46
Topical exclusions: 22


In [4]:

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"PATSTAT client initialized: {ENV}")


PATSTAT client initialized: PROD



## 6. Define the family population

The analysis is conducted at the DOCDB family level.

Families are identified using positive
`docdb_family_id` values.

Only applications with the selected intellectual
property right type are considered.

The family year is defined as the minimum
`earliest_filing_year` across these applications.

Families are retained when their family year
falls between 2000 and 2022, inclusive.

This follows the temporal convention used in the
original Step 0 notebook.


In [5]:

family_year = func.min(
    TLS201_APPLN.earliest_filing_year
)

family_scope = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        family_year.label("family_year")
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE,
        TLS201_APPLN.docdb_family_id > 0
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .having(
        family_year.between(YEAR_MIN, YEAR_MAX)
    )
    .subquery()
)

print("Family scope prepared.")


Family scope prepared.



## 7. Collect family-level keyword evidence

Keyword evidence is aggregated separately for each
DOCDB patent family.

Five Boolean indicators are generated:

1. Specific Precision Agriculture terminology.
2. Agricultural terminology.
3. Standard technological terminology.
4. Generic platform terminology.
5. Additional precision-related terminology.

Each indicator is true if at least one application
belonging to the family contains a matching term.

Evidence from different applications may contribute
to the same family-level retrieval decision.

All five keyword indicators are computed together
rather than through separate application-level
correlated subqueries.


In [6]:

keyword_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            keyword_condition(SPECIFIC_PATTERNS)
        ).label("specific_phrase"),

        func.logical_or(
            keyword_condition(ENABLING_PATTERNS)
        ).label("enabling_technology"),

        func.logical_or(
            keyword_condition(PA_FUNCTION_PATTERNS)
        ).label("pa_function"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id
        == family_scope.c.fam_id
    )
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id
        == TLS202_APPLN_TITLE.appln_id
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id
        == TLS203_APPLN_ABSTR.appln_id
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("Family-level keyword flags prepared.")


Family-level keyword flags prepared.



## 8. Collect CPC and IPC evidence

Patent classification evidence is collected separately
from textual evidence.

This avoids multiplying keyword observations when
an application has multiple CPC or IPC classifications.

For each DOCDB family, we identify whether any
application has:

- A highly specific PA CPC code.
- An agricultural-context CPC code.
- A highly specific PA IPC code, if configured.
- An agricultural-context IPC code.

CPC and IPC information is used for direct inclusion
through R1 and to establish agricultural context
within R3.


In [7]:

cpc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SPECIFIC_CPC
            )
        ).label("specific_cpc"),

        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                AGRICULTURAL_CPC
            )
        ).label("agricultural_cpc"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id
        == family_scope.c.fam_id
    )
    .join(
        TLS224_APPLN_CPC,
        TLS201_APPLN.appln_id
        == TLS224_APPLN_CPC.appln_id
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("CPC flags prepared.")


CPC flags prepared.


In [8]:

ipc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),

        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                SPECIFIC_IPC
            )
        ).label("specific_ipc"),

        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                AGRICULTURAL_IPC
            )
        ).label("agricultural_ipc"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id
        == family_scope.c.fam_id
    )
    .join(
        TLS209_APPLN_IPC,
        TLS201_APPLN.appln_id
        == TLS209_APPLN_IPC.appln_id
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .group_by(
        TLS201_APPLN.docdb_family_id
    )
    .subquery()
)

print("IPC flags prepared.")


IPC flags prepared.


In [9]:

excluded_queries = []

if EXCLUDE_CPC:
    excluded_queries.append(
        db.query(
            TLS201_APPLN.docdb_family_id.label("fam_id")
        )
        .select_from(TLS201_APPLN)
        .join(
            TLS224_APPLN_CPC,
            TLS201_APPLN.appln_id
            == TLS224_APPLN_CPC.appln_id
        )
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE,
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                EXCLUDE_CPC
            )
        )
    )

if EXCLUDE_IPC:
    excluded_queries.append(
        db.query(
            TLS201_APPLN.docdb_family_id.label("fam_id")
        )
        .select_from(TLS201_APPLN)
        .join(
            TLS209_APPLN_IPC,
            TLS201_APPLN.appln_id
            == TLS209_APPLN_IPC.appln_id
        )
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE,
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                EXCLUDE_IPC
            )
        )
    )

if excluded_queries:
    excluded_query = excluded_queries[0]

    for query in excluded_queries[1:]:
        excluded_query = excluded_query.union(query)

    excluded_families = excluded_query.distinct().subquery()
else:
    excluded_families = None

print("Classification exclusions prepared.")


Classification exclusions prepared.


In [10]:
# TITLE-ONLY thematic exclusions (forestry, golf/turf, livestock, gardening)
# The title is checked rather than the full abstract to reduce incidental exclusions.
# This is a family-level flag: any application title in the DOCDB family may trigger it.
if EXCLUSION_PATTERNS:
    title_exclusion_pattern = "|".join(
        f"(?:{pattern})" for pattern in EXCLUSION_PATTERNS.values()
    )
    topical_exclusions = (
        db.query(TLS201_APPLN.docdb_family_id.label("fam_id"))
        .select_from(TLS201_APPLN)
        .join(family_scope, TLS201_APPLN.docdb_family_id == family_scope.c.fam_id)
        .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .filter(
            TLS201_APPLN.ipr_type == IPR_TYPE,
            func.regexp_contains(
                func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, "")),
                title_exclusion_pattern,
            ),
        )
        .distinct()
        .subquery()
    )
else:
    topical_exclusions = None

# Protected CPC: A01B79/005 ALWAYS survives the thematic exclusions.
protected_families = (
    db.query(TLS201_APPLN.docdb_family_id.label("fam_id"))
    .select_from(TLS201_APPLN)
    .join(family_scope, TLS201_APPLN.docdb_family_id == family_scope.c.fam_id)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE,
        symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, ["A01B79/005"]),
    )
    .distinct()
    .subquery()
)
print("Topical exclusions and protected CPC prepared.")

Topical exclusions and protected CPC prepared.



## 10. Combine family-level evidence

Textual evidence and patent classification evidence
are combined using the DOCDB family identifier.

Missing indicators are interpreted as false.

The resulting family-level record contains:

- Earliest filing year.
- Specific PA keyword evidence.
- Agricultural keyword evidence.
- Standard technological evidence.
- Generic platform evidence.
- Additional precision-related evidence.
- Specific and agricultural CPC/IPC evidence.

These indicators will be combined into the three
retrieval routes in the following step.


In [11]:

flags = (
    db.query(
        family_scope.c.fam_id,
        family_scope.c.family_year,

        func.coalesce(
            keyword_flags.c.specific_phrase, False
        ).label("specific_phrase"),

        func.coalesce(
            keyword_flags.c.enabling_technology, False
        ).label("enabling_technology"),

        func.coalesce(
            keyword_flags.c.pa_function, False
        ).label("pa_function"),

        func.coalesce(
            cpc_flags.c.specific_cpc, False
        ).label("specific_cpc"),

        func.coalesce(
            cpc_flags.c.agricultural_cpc, False
        ).label("agricultural_cpc"),

        func.coalesce(
            ipc_flags.c.specific_ipc, False
        ).label("specific_ipc"),

        func.coalesce(
            ipc_flags.c.agricultural_ipc, False
        ).label("agricultural_ipc"),
    )
    .select_from(family_scope)
    .outerjoin(
        keyword_flags,
        family_scope.c.fam_id
        == keyword_flags.c.fam_id
    )
    .outerjoin(
        cpc_flags,
        family_scope.c.fam_id
        == cpc_flags.c.fam_id
    )
    .outerjoin(
        ipc_flags,
        family_scope.c.fam_id
        == ipc_flags.c.fam_id
    )
    .subquery()
)

print("Combined family-level evidence prepared.")


Combined family-level evidence prepared.


## Three-route selection

R1 and R2 are direct; R3 requires agricultural CPC/IPC **and** an enabling-technology term **and** a PA-function term. Existing classification exclusions constrain R3, while topical exclusions constrain all routes except protected `A01B79/005` families.

In [12]:
# THREE ROUTES — same family-level SQL architecture as Step0_PA_v9
R1 = or_(flags.c.specific_cpc, flags.c.specific_ipc)
R2 = flags.c.specific_phrase

agricultural_class = or_(flags.c.agricultural_cpc, flags.c.agricultural_ipc)

# Unified R3: agricultural classification + enabling technology + PA function.
r3_base = and_(
    agricultural_class,
    flags.c.enabling_technology,
    flags.c.pa_function,
)

# Preserve existing CPC/IPC exclusions only for the technological route.
if excluded_families is not None:
    R3 = and_(
        r3_base,
        ~flags.c.fam_id.in_(db.query(excluded_families.c.fam_id)),
    )
else:
    R3 = r3_base

# Thematic title exclusions apply to all routes, but never exclude A01B79/005.
if topical_exclusions is not None:
    topical_allowed = or_(
        ~flags.c.fam_id.in_(db.query(topical_exclusions.c.fam_id)),
        flags.c.fam_id.in_(db.query(protected_families.c.fam_id)),
    )
else:
    topical_allowed = True

sector_query = (
    db.query(
        *flags.c,
        R1.label("route_1"),
        R2.label("route_2"),
        R3.label("route_3"),
    )
    .select_from(flags)
    .filter(and_(or_(R1, R2, R3), topical_allowed))
)
selected_families = sector_query.subquery()
print("Final family-level query: R1 OR R2 OR R3")

Final family-level query: R1 OR R2 OR R3


In [13]:
# Old diagnostic removed — use the three-route counts in the next cell.

In [14]:
# Count UNIQUE DOCDB families for each route combination
route_counts_query = (
    db.query(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
        func.count(func.distinct(selected_families.c.fam_id)).label("n_families"),
    )
    .select_from(selected_families)
    .group_by(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
    )
    .order_by(func.count(func.distinct(selected_families.c.fam_id)).desc())
)
route_counts = patstat.df(route_counts_query)
if route_counts.empty:
    raise ValueError("No patent families retrieved")
total_families = int(route_counts["n_families"].sum())
route_counts["share_pct"] = (100 * route_counts["n_families"] / total_families).round(1)
print(f"Total DOCDB families: {total_families:,}")
display(route_counts)


Total DOCDB families: 5,524


,route_1,route_2,route_3,n_families,share_pct
0,True,False,False,3300,59.7
1,False,False,True,1495,27.1
2,False,True,False,309,5.6
3,True,False,True,307,5.6
4,False,True,True,54,1.0
5,True,True,False,50,0.9
6,True,True,True,9,0.2


In [15]:
# Route totals and incremental contributions (overlaps are expected)
for route in ["route_1", "route_2", "route_3"]:
    n = int(route_counts.loc[route_counts[route] == True, "n_families"].sum())
    exclusive = int(route_counts.loc[
        (route_counts[route] == True) &
        route_counts[[r for r in ["route_1", "route_2", "route_3"] if r != route]].eq(False).all(axis=1),
        "n_families"
    ].sum())
    print(f"{route}: {n:,} families; route-exclusive: {exclusive:,}")


route_1: 3,666 families; route-exclusive: 3,300
route_2: 422 families; route-exclusive: 309
route_3: 1,865 families; route-exclusive: 1,495



# Manual validation

## 13. Random sampling

A simple random sample of 100 DOCDB patent families
is selected from the retrieved population.

Sampling is performed without replacement.

Each family has the same probability of selection.

A fixed random seed ensures reproducibility when
the underlying population remains unchanged.

Manual validation assesses whether the retrieved
inventions are relevant to the Precision Agriculture
sector.

The following categories are used:

- **A — Core PA:** Innovations directly implementing
  precision-agriculture functions.
- **B — Supporting PA:** Technologies directly supporting
  precision-agriculture functions.
- **C — Out of scope:** Generic agricultural machinery
  improvements or unrelated inventions.

The final binary relevance label is:

- 1 = Core or Supporting Precision Agriculture.
- 0 = Out of scope.

Retrieval-route indicators are not used as automatic
relevance labels.


In [16]:

sampling_query = (
    db.query(
        selected_families.c.fam_id
    )
    .select_from(selected_families)
    .distinct()
)

sampling_population = patstat.df(sampling_query)

if sampling_population.empty:
    raise ValueError("No families available for sampling.")

if sampling_population["fam_id"].isna().any():
    raise ValueError("Missing family identifiers.")

sampling_population["fam_id"] = (
    sampling_population["fam_id"].astype("int64")
)

sampling_population = (
    sampling_population
    .drop_duplicates("fam_id")
    .sort_values("fam_id")
    .reset_index(drop=True)
)

assert len(sampling_population) == total_families

print(
    f"Families available for sampling: "
    f"{len(sampling_population):,}"
)

display(sampling_population.head())


Families available for sampling: 5,524


,fam_id
0,3828303
1,3828568
2,3831289
3,3945251
4,7628526


In [17]:

if not 0 < SAMPLE_SIZE <= len(sampling_population):
    raise ValueError("Invalid sample size.")

validation_sample = (
    sampling_population
    .sample(
        n=SAMPLE_SIZE,
        replace=False,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

assert len(validation_sample) == SAMPLE_SIZE
assert validation_sample["fam_id"].is_unique

print(f"Sampled families: {len(validation_sample)}")
display(validation_sample.head())


Sampled families: 100


,fam_id
0,89379804
1,76518337
2,70918265
3,44509278
4,59020674


In [18]:

sample_ids = (
    validation_sample["fam_id"]
    .astype(int)
    .tolist()
)

title_present = (
    func.length(
        func.trim(
            func.coalesce(
                TLS202_APPLN_TITLE.appln_title, ""
            )
        )
    ) > 0
)

abstract_present = (
    func.length(
        func.trim(
            func.coalesce(
                TLS203_APPLN_ABSTR.appln_abstract, ""
            )
        )
    ) > 0
)

english_title = and_(
    TLS202_APPLN_TITLE.appln_title_lg == "en",
    title_present
)

english_abstract = and_(
    TLS203_APPLN_ABSTR.appln_abstract_lg == "en",
    abstract_present
)

text_priority = case(
    (and_(english_abstract, english_title), 0),
    (english_abstract, 1),
    (and_(abstract_present, english_title), 2),
    (abstract_present, 3),
    (english_title, 4),
    (title_present, 5),
    else_=6
)

representative_rank = func.row_number().over(
    partition_by=TLS201_APPLN.docdb_family_id,
    order_by=[
        text_priority,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_id
    ]
).label("representative_rank")

ranked_applications = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        TLS201_APPLN.appln_id,
        TLS202_APPLN_TITLE.appln_title.label("title"),
        TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
        representative_rank
    )
    .select_from(TLS201_APPLN)
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id
        == TLS202_APPLN_TITLE.appln_id
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id
        == TLS203_APPLN_ABSTR.appln_id
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(sample_ids),
        TLS201_APPLN.ipr_type == IPR_TYPE
    )
    .subquery()
)

representatives_query = (
    db.query(
        ranked_applications.c.fam_id,
        ranked_applications.c.appln_id,
        ranked_applications.c.title,
        ranked_applications.c.abstract
    )
    .select_from(ranked_applications)
    .filter(
        ranked_applications.c.representative_rank == 1
    )
)

representatives = patstat.df(representatives_query)

print(
    "Representative applications retrieved:",
    len(representatives)
)


Representative applications retrieved: 100


In [19]:
# Duplicate representative-text cell removed. Continue to the next cell.


In [20]:

if representatives.empty:
    raise ValueError(
        "No representative applications retrieved."
    )

representatives["fam_id"] = (
    representatives["fam_id"].astype("int64")
)

validation_sheet = (
    validation_sample[["fam_id"]]
    .merge(
        representatives,
        on="fam_id",
        how="left",
        validate="one_to_one",
        sort=False
    )
)

if validation_sheet["appln_id"].isna().any():
    raise ValueError(
        "Some sampled families have no application."
    )

validation_sheet["appln_id"] = (
    validation_sheet["appln_id"].astype("int64")
)

validation_sheet[["title", "abstract"]] = (
    validation_sheet[["title", "abstract"]]
    .fillna("")
)

# Empty columns for manual validation
validation_sheet["category"] = ""
validation_sheet["label"] = ""
validation_sheet["reason"] = ""

assert len(validation_sheet) == SAMPLE_SIZE
assert validation_sheet["fam_id"].is_unique

print("Validation table ready:", len(validation_sheet))

display(validation_sheet.head())


Validation table ready: 100


,fam_id,appln_id,title,abstract,category,label,reason
0,89379804,603872629,WORK ASSIST SYSTEM,A work assist system capable of improving conv...,,,
1,76518337,553018861,Rice seed-metering device capable of automatic...,The invention discloses a rice seed-metering d...,,,
2,70918265,542270415,Sensor arrangement for an agricultural vehicle,A sensor arrangement for an agricultural vehic...,,,
3,44509278,335775390,METHOD FOR APPLYING AGRICULTURAL SUBSTANCES IN...,The invention relates to a method for applying...,,,
4,59020674,480035249,Generating digital models of relative yield of...,A computer implemented method for generating d...,,,


In [21]:
# Add route membership for each sampled family (one SQL row per family)
route_sample_query = (
    db.query(
        selected_families.c.fam_id,
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
    )
    .select_from(selected_families)
    .filter(selected_families.c.fam_id.in_(sample_ids))
)
route_sample = patstat.df(route_sample_query)
route_sample["fam_id"] = route_sample["fam_id"].astype("int64")
route_sample = route_sample.drop_duplicates("fam_id")
validation_sheet = validation_sheet.merge(route_sample, on="fam_id", validate="one_to_one")
assert len(validation_sheet) == SAMPLE_SIZE
print("Route flags attached to validation sheet")


Route flags attached to validation sheet


In [22]:
validation_path = OUTPUT_DIR / "hand_validation_PA_v10_merged.csv"
validation_sheet.to_csv(validation_path, index=False, encoding="utf-8-sig")
print("Validation CSV:", validation_path)
print("Sample size:", len(validation_sheet))


Validation CSV: /home/jovyan/team-38-PA/notebooks/outputs/step0/hand_validation_PA_v10_merged.csv
Sample size: 100


In [25]:
import pandas as pd

df = pd.read_csv(
    "outputs/step0/hand_validation_PA_v10_merged_labeled.csv"
)

r1_c = df[
    (df["category"] == "C") &
    (df["route_1"] == True) &
    (df["route_2"] == False) &
    (df["route_3"] == False)
]
print("Colonne disponibili:", df.columns.tolist())

id_col = next(
    col for col in ["family_id", "fam_id", "docdb_family_id"]
    if col in df.columns
)

print(r1_c[[id_col, "title"]].to_string(index=False))

Colonne disponibili: ['fam_id', 'appln_id', 'title', 'abstract', 'category', 'label', 'reason', 'route_1', 'route_2', 'route_3', 'review_needed', 'relevant']
  fam_id                                                                   title
84798480         Advanced movement through vegetation with an autonomous vehicle
33154565 Method and apparatus for determining fertilizer requirements in gardens
69650549                               METHOD FOR OPERATING AN AUTOMATED VEHICLE
45973789                                     Exchanging water allocation credits
45525420                                     Seed retrieval apparatus and method
40623216                        Multi-antenna GNSS positioning method and system


In [29]:
import pandas as pd
from sqlalchemy import text, bindparam

# Nove famiglie C del secondo campione PA_v10
family_ids = [
    89379804,
    65236797,
    75787299,
    43622802,
    53269303,
    44860478,
    74488141,
    43797885,
    60957001,
]

sql = text("""
SELECT
    a.docdb_family_id AS fam_id,
    MAX(
        CASE
            WHEN REPLACE(c.cpc_class_symbol, ' ', '') = 'A01B79/005'
            THEN 1
            ELSE 0
        END
    ) AS has_protected_cpc
FROM tls201_appln a
LEFT JOIN tls224_appln_cpc c
    ON a.appln_id = c.appln_id
WHERE a.docdb_family_id IN :family_ids
GROUP BY a.docdb_family_id
ORDER BY a.docdb_family_id
""").bindparams(bindparam("family_ids", expanding=True))

result = db.execute(
    sql,
    {"family_ids": family_ids}
)

df_cpc = pd.DataFrame(
    result.fetchall(),
    columns=["fam_id", "has_protected_cpc"]
)

display(df_cpc)

,fam_id,has_protected_cpc
0,43622802,0
1,43797885,1
2,44860478,0
3,53269303,1
4,60957001,0
5,65236797,1
6,74488141,1
7,75787299,1
8,89379804,0
